## Extension Analysis Work (on-going)

In [2]:
from tabulate import tabulate

In [3]:
# Functions to parse rapl data
def load_workflow_rapl_readings(cluster, workflow):
    with open(f'nxf-experiments/{cluster}-cluster/{workflow}-runs.csv', 'r') as file:
        data = [line.strip().split(',') for line in file.readlines()]
    
    return data[1:]

def process_rapl_data(data):
    processed = {}
    run = 1

    for row in data:
        processed[run] = {}
        processed[run]['pkg'] = float(row[1])
        processed[run]['dram'] = float(row[2])
        processed[run]['total'] = float(row[3])
        run += 1

    return processed

In [4]:
# Functions to parse ichnos data
def load_ichnos_summary_file(summary_file):
    with open(summary_file, 'r') as file:
        raw = [line.strip() for line in file.readlines()]

    cpu = float(raw[7].split(':')[1].strip()[:-3])
    task_mem = float(raw[9].split(':')[1].strip()[:-3])
    node_mem = float(raw[15].split(':')[1].strip()[:-3])

    return (cpu, task_mem, node_mem)

def load_ichnos_data(cluster, workflow, runs, strategy):
    workflow_path = f'nxf-experiments/{cluster}-cluster/{workflow}'
    data = {}

    for run in range(1, runs + 1):
        data[run] = {}
        (cpu, task_mem, node_mem) = load_ichnos_summary_file(f'{workflow_path}/{run}/{cluster}-{workflow}-{run}-1-{strategy}-summary.txt')
        data[run]['pkg'] = cpu
        data[run]['dram'] = task_mem + node_mem 
        data[run]['total'] = cpu + task_mem + node_mem

    return data

In [5]:
# Generic functions
def get_error(experimental, actual):
    return round((abs(experimental - actual) / actual) * 100, 2)

In [6]:
print('HU Cluster Workflow Executions')
rapl_data = {'hu': {}, 'gu': {}}
rapl_data['hu']['rnaseq'] = process_rapl_data(load_workflow_rapl_readings('hu', 'rnaseq'))
rapl_data['hu']['chipseq'] = process_rapl_data(load_workflow_rapl_readings('hu', 'chipseq'))
rapl_data['gu']['rnaseq'] = process_rapl_data(load_workflow_rapl_readings('gu', 'rnaseq'))

ichnos_data = {'hu': {}, 'gu': {}}
ichnos_data['hu']['rnaseq'] = load_ichnos_data('hu', 'rnaseq', len(rapl_data['hu']['rnaseq']), 'schedutil_linear')
ichnos_data['hu']['chipseq'] = load_ichnos_data('hu', 'chipseq', len(rapl_data['hu']['chipseq']), 'schedutil_linear')
ichnos_data['gu']['rnaseq'] = load_ichnos_data('gu', 'rnaseq', len(rapl_data['gu']['rnaseq']), 'ondemand_linear')

table_data = []
table_head = ['cluster', 'workflow', 'run', 'ichnos (kWh)', 'rapl (kWh)', 'error (%)']

for cluster in ['hu', 'gu']:
    for workflow in rapl_data[cluster].keys():
        for run in range(1, len(rapl_data[cluster][workflow].keys()) + 1):
            rapl_entry = rapl_data[cluster][workflow][run]
            ichnos_entry = ichnos_data[cluster][workflow][run]
            table_data.append([cluster, workflow, run, round(ichnos_entry['total'], 2), round(rapl_entry['total'], 2), get_error(ichnos_entry['total'], rapl_entry['total'])])

print(tabulate(table_data, table_head))

HU Cluster Workflow Executions
cluster    workflow      run    ichnos (kWh)    rapl (kWh)    error (%)
---------  ----------  -----  --------------  ------------  -----------
hu         rnaseq          1            2.17          2.55        14.93
hu         rnaseq          2            2.16          2.6         17.1
hu         rnaseq          3            2.17          1.7         27.53
hu         rnaseq          4            2.27          2.11         7.13
hu         rnaseq          5            2.27          2.44         6.77
hu         chipseq         1            3.66          3.85         5.1
hu         chipseq         2            3.68          3.87         4.9
hu         chipseq         3            3.7           3.46         6.98
gu         rnaseq          1            1.59          1.78        10.76
